# Bài 1: Kiểm chứng tin tức (notebook mẫu)

Mỗi mục của tập test gồm một **nhận định** và mã của **bài báo** mà nhận định nói tới. Với mỗi nhận định, cần dự đoán một
trong ba nhãn. Nhãn `SUPPORTED` nghĩa là bài báo xác nhận nhận định, nhãn `REFUTED` nghĩa là bài báo bác bỏ nhận định, và nhãn
`NEI` nghĩa là bài báo không có đủ thông tin để kết luận. Điểm là macro-F1, tức trung bình cộng điểm F1 của ba nhãn.

Bài có hai giai đoạn. Lúc đầu chỉ có `public_test.csv` và `articles.csv` (bài báo của train và public test). Cuối giờ thi, ban
tổ chức công bố mật khẩu của `private_test.zip`. Giải nén tệp đó **vào cùng thư mục với notebook này**, rồi chạy lại toàn bộ
notebook (Restart & Run All). Notebook sẽ tự chuyển sang `private_test.csv` và `private_articles.csv`.

```
unzip private_test.zip
```

Baseline trong notebook này làm ba bước:

1. Chia mỗi bài báo thành từng câu, rồi dùng TF-IDF để tìm câu giống nhận định nhất. TF-IDF biểu diễn mỗi câu bằng trọng số
   của các từ trong câu, trong đó từ càng hiếm thì trọng số càng cao.
2. Tính mười đặc trưng dạng số cho mỗi nhận định. Các đặc trưng gồm độ giống giữa nhận định và câu tìm được, tỉ lệ từ của
   nhận định có mặt trong bài báo, số con số có trong nhận định nhưng không có trong bài báo, chênh lệch số từ phủ định (như
   "không", "chưa") giữa nhận định và câu tìm được, và độ dài của nhận định.
3. Train model hồi quy logistic trên mười đặc trưng đó để dự đoán nhãn.

Cả notebook chạy trên CPU trong khoảng nửa phút. Ban tổ chức đo được macro-F1 khoảng **0,57** trên validation và khoảng
**0,60** trên tập test.

**Quy định khi làm bài**

- Chỉ dùng Python và các thư viện trong danh sách của thể lệ. Không cài thêm thư viện bằng `pip` hay `conda`.
- Được dùng các model pretrained cơ bản do chính các thư viện đó cung cấp, ví dụ vector từ của fastText hay gensim. Không
  dùng model ngôn ngữ pretrained dạng Transformer như PhoBERT, XLM-R, multilingual-e5, ViT5, và không dùng LLM.
- Tệp kết quả phải được tạo ra bằng cách chạy notebook tuần tự từ ô đầu tiên đến ô cuối cùng.
- Không sửa tay dữ liệu đầu vào hay tệp kết quả.
- Không học TF-IDF hay bất kỳ model nào trên văn bản của public test hoặc private test.
- Tệp nộp là một tệp zip chứa đúng một tệp CSV hai cột `id`, `label`, đủ mọi `id` của tập test **của giai đoạn đang chạy**.


In [ ]:
import re, sys, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path(".")  # thư mục chứa notebook, train.csv, articles.csv, public_test.csv
sys.path.insert(0, str(DATA_DIR))
from scorer import score

train = pd.read_csv(DATA_DIR / "train.csv")
articles = pd.read_csv(DATA_DIR / "articles.csv")

# Giai đoạn private: giải nén private_test.zip vào DATA_DIR rồi Restart & Run All.
if (DATA_DIR / "private_test.csv").exists():
    test = pd.read_csv(DATA_DIR / "private_test.csv")
    extra = pd.read_csv(DATA_DIR / "private_articles.csv")
    articles = pd.concat([articles, extra], ignore_index=True).drop_duplicates("article_id")
    phase = "private"
else:
    test = pd.read_csv(DATA_DIR / "public_test.csv")
    phase = "public"

print("giai đoạn", phase)
print(len(train), "nhận định train,", len(test), "nhận định test,", len(articles), "bài báo")
print(train.label.value_counts())


## 1. Đặc trưng và model

Ô dưới đây định nghĩa các hàm tách câu, tính đặc trưng và train model. TF-IDF chỉ được học từ bài báo và nhận định của phần
dữ liệu dùng để train. Nếu học cả trên bài báo của validation, điểm validation sẽ cao hơn điểm thật. Nếu học trên văn bản của
tập test (public hoặc private), đội thi vi phạm thể lệ.


In [ ]:
def tach_cau(van_ban):
    """Tách bài báo thành câu theo dấu kết thúc câu và bỏ các mảnh ngắn hơn ba từ."""
    cau = [c for c in re.split(r"(?<=[.!?…])\s+", str(van_ban)) if len(c.split()) >= 3]
    return cau or [str(van_ban)]


def tu(s):
    return re.findall(r"\w+", str(s).lower())


PHU_DINH = {"không", "chưa", "chẳng", "chả", "đâu"}
SO = re.compile(r"\d+(?:[.,]\d+)*")


def tao_dac_trung(df, cau_cua_bai, vec, ma_tran):
    """So mỗi nhận định với câu giống nó nhất trong bài báo và với toàn bài; trả về ma trận 10 đặc trưng."""
    rows = []
    for claim, a in zip(df.claim, df.article_id):
        sims = (ma_tran[a] @ vec.transform([claim]).T).toarray().ravel()
        best = int(sims.argmax())
        top = np.sort(sims)[::-1]
        ct, bt = tu(claim), tu(cau_cua_bai[a][best])
        toan_bai = set(w for c in cau_cua_bai[a] for w in tu(c))
        cb, bb = set(zip(ct, ct[1:])), set(zip(bt, bt[1:]))
        so_claim, so_bai = set(SO.findall(claim)), set(SO.findall(" ".join(cau_cua_bai[a])))
        rows.append([top[0], top[:3].mean(),
                     np.mean([w in toan_bai for w in ct]), np.mean([w in set(bt) for w in ct]),
                     len(cb & bb) / max(1, len(cb)), len(so_claim - so_bai), len(so_claim),
                     sum(w in PHU_DINH for w in ct) - sum(w in PHU_DINH for w in bt),
                     len(ct), len(ct) / max(1, len(bt))])
    return np.array(rows)


def huan_luyen(df_train, cau_cua_bai):
    """Học TF-IDF từ phần train, tính đặc trưng rồi train hồi quy logistic."""
    vec = TfidfVectorizer(tokenizer=tu, token_pattern=None, lowercase=False, ngram_range=(1, 2),
                          sublinear_tf=True, min_df=2)
    vec.fit([c for a in set(df_train.article_id) for c in cau_cua_bai[a]] + list(df_train.claim))
    ma_tran = {a: vec.transform(c) for a, c in cau_cua_bai.items()}
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(tao_dac_trung(df_train, cau_cua_bai, vec, ma_tran), df_train.label)
    return model, vec, ma_tran


cau_cua_bai = {r.article_id: tach_cau(r.text) for r in articles.itertuples()}


## 2. Tự đánh giá trên validation

Notebook giữ lại khoảng 20 % số bài báo của train làm validation. Validation được tách **theo bài báo**, nghĩa là mọi nhận
định của cùng một bài báo đều nằm ở phần train hoặc đều nằm ở validation. Ban tổ chức cũng chia train và tập test theo cách
này, nên điểm validation gần với điểm trên tập test.


In [ ]:
tach = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
i_tr, i_va = next(tach.split(train, groups=train.article_id))
tr, va = train.iloc[i_tr], train.iloc[i_va]
model, vec, ma_tran = huan_luyen(tr, cau_cua_bai)
du_doan_va = va[["id"]].assign(label=model.predict(tao_dac_trung(va, cau_cua_bai, vec, ma_tran)))
print("macro-F1 trên validation:", round(score(va[["id", "label"]], du_doan_va, "id"), 4))


## 3. Dự đoán tập test

Ô dưới đây train lại model trên toàn bộ train rồi ghi tệp `bai1_submission.csv` cho giai đoạn đang chạy (public hoặc private).


In [ ]:
model, vec, ma_tran = huan_luyen(train, cau_cua_bai)
nop = test[["id"]].assign(label=model.predict(tao_dac_trung(test, cau_cua_bai, vec, ma_tran)))
nop.to_csv("bai1_submission.csv", index=False)
print("Đã ghi bai1_submission.csv:", len(nop), "dòng, giai đoạn", phase)
print(nop.label.value_counts())


## 4. Nộp bài

Tệp nộp là một tệp zip chứa đúng một tệp CSV, là `bai1_submission.csv` vừa ghi ra. Ô dưới đây tạo `submission.zip` sẵn để nộp.

CSV trong zip có hai cột `id` và `label`, đủ mọi `id` của tập test giai đoạn hiện tại (`public_test.csv` hoặc `private_test.csv`).
Nhãn chỉ nhận `SUPPORTED`, `REFUTED` hoặc `NEI`.


In [ ]:
csv_path = Path("bai1_submission.csv")
zip_path = Path("submission.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, arcname=csv_path.name)
print("Đã ghi", zip_path.resolve())
